In [1]:
import json
import re
from typing import Any
import math

import geopandas as gpd
import pandas as pd
from unidecode import unidecode

In [2]:
sectors = gpd.read_file('source_data/sectors2025.gpkg')

In [3]:
sectors.columns

Index(['CS01012025', 'T_SEC_NL', 'T_SEC_FR', 'T_SEC_DE', 'C_NIS7', 'C_NIS6',
       'T_NIS6_NL', 'T_NIS6_FR', 'MUN_DISTR', 'CNIS5_2025', 'T_MUN_NL',
       'T_MUN_FR', 'T_MUN_DE', 'CNIS_ARRD_', 'T_ARRD_NL', 'T_ARRD_FR',
       'T_ARRD_DE', 'CNIS_PROVI', 'T_PROVI_NL', 'T_PROVI_FR', 'T_PROVI_DE',
       'CNIS_REGIO', 'T_REGIO_NL', 'T_REGIO_FR', 'T_REGIO_DE', 'C_COUNTRY',
       'NUTS1_2021', 'NUTS2_2021', 'NUTS3_2021', 'M_AREA_HA', 'M_PERI_M',
       'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='object')

In [4]:
def simplify_geometry(gdf: gpd.GeoDataFrame, tolerance_in_meters: float) -> gpd.GeoDataFrame:
    # compute the tolerance. If the crs is not WGS84, assume that 1 unit = 1m. if the crs is not specified, assume it is WGS84
    if gdf.crs is None or gdf.crs.to_epsg() == 4326 or gdf.crs.to_epsg() == 4979:
    	meters_per_degree = 111120
    else:
    	meters_per_degree = 1
    
    tolerance = tolerance_in_meters / meters_per_degree

    new_gdf = gdf.copy()
    new_gdf.geometry = new_gdf.geometry.simplify(tolerance)

    return new_gdf


def create_aggregate(nis_col: str, tolerance: float, sectors: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Dissolve statistical sectors into municipalities."""
    print("Dissolving...")

    # Dissolve sectors
    new_gdf = sectors.dissolve(by=nis_col, as_index=False).rename(columns={nis_col: 'nis'})

    # Keep relevant columns
    keep_cols = ['nis', "geometry"]
    new_gdf = new_gdf[keep_cols]
    

    print(f"Created new gpd with {len(new_gdf)} elements")

    # reproject to WGS84
    print("Reprojecting...")
    new_gdf = new_gdf.to_crs(epsg=4326)
    
    # Further simplify after reprojection (smaller tolerance in degrees)
    print("Simplifying...")
    new_gdf = simplify_geometry(new_gdf, tolerance)

    
    return new_gdf

In [5]:
# Create each administrative level
municipalities = create_aggregate("CNIS5_2025", 50, sectors)
provinces = create_aggregate("CNIS_PROVI", 200, sectors)
regions = create_aggregate("CNIS_REGIO", 500, sectors)

# Save GeoJSON files
print("\nSaving GeoJSON files...")
municipalities.to_file("../geodata/municipalities.geojson", driver='GeoJSON', coordinate_precision=4)
provinces.to_file("../geodata/provinces.geojson", driver='GeoJSON', coordinate_precision=3)
regions.to_file("../geodata/regions.geojson", driver='GeoJSON', coordinate_precision=3)


Dissolving...
Created new gpd with 565 elements
Reprojecting...
Simplifying...
Dissolving...
Created new gpd with 10 elements
Reprojecting...
Simplifying...
Dissolving...
Created new gpd with 3 elements
Reprojecting...
Simplifying...

Saving GeoJSON files...
